In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StringIndexer, StandardScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import mlflow
import mlflow.spark

spark = SparkSession.builder.getOrCreate()
spark.sql("USE fraud_db")

print("✅ All libraries loaded!")
print(f"   MLflow version : {mlflow.__version__}")

In [0]:

silver_df = spark.table("fraud_db.silver_transactions")

feature_cols = [
    "amount",
    "hour_of_day",
    "login_attempts",
    "is_night_transaction",
    "is_high_amount",
    "is_suspicious_country",
    "is_multiple_attempts",
    "is_online_or_atm",
    "fraud_risk_score"
]

label_col = "is_fraud"

ml_df = silver_df.select(feature_cols + [label_col]) \
                 .na.drop()    
                 
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

print(f"✅ Dataset prepared!")
print(f"   Total records  : {ml_df.count():,}")
print(f"   Training set   : {train_df.count():,} rows (80%)")
print(f"   Test set       : {test_df.count():,} rows (20%)")
print(f"   Feature count  : {len(feature_cols)}")
print(f"\n   Features used  : {feature_cols}")

# Check class balance
fraud  = ml_df.filter(col("is_fraud") == 1).count()
legit  = ml_df.filter(col("is_fraud") == 0).count()
print(f"\n   🚨 Fraud (1)   : {fraud:,} ({round(fraud/ml_df.count()*100,1)}%)")
print(f"   ✅ Legit  (0)  : {legit:,} ({round(legit/ml_df.count()*100,1)}%)")

In [0]:
# Stage 1: Combine all feature columns into one vector
assembler = VectorAssembler(
    inputCols  = feature_cols,
    outputCol  = "raw_features",
    handleInvalid = "skip"      # skip rows with bad values
)

# Stage 2: Scale features to same range
# WHY: amount (5-9999) vs is_fraud (0-1) — huge scale gap.
# Random Forest doesn't need this, but it's good practice.
scaler = StandardScaler(
    inputCol  = "raw_features",
    outputCol = "features",
    withMean  = True,
    withStd   = True
)

# Stage 3: Random Forest Classifier
# WHY Random Forest: handles imbalanced classes well,
# gives feature importance, robust to outliers, no
# need to normalize (but we do it for learning purposes).
rf = RandomForestClassifier(
    labelCol    = "is_fraud",
    featuresCol = "features",
    numTrees    = 50,           # 50 decision trees
    maxDepth    = 6,            # max depth per tree
    seed        = 42
)

# Assemble the pipeline
pipeline = Pipeline(stages=[assembler, scaler, rf])

print("✅ ML Pipeline built!")
print("   Stage 1 → VectorAssembler  (combine features)")
print("   Stage 2 → StandardScaler   (normalize scale)")
print("   Stage 3 → RandomForest     (50 trees, depth 6)")

In [0]:

from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_experiment("/fraud_detection_experiment")

with mlflow.start_run(run_name="RandomForest_v1") as run:

    RUN_ID = run.info.run_id
    print(f"🚀 MLflow Run Started!")
    print(f"   Run ID : {RUN_ID}\n")

    # ── Log Parameters ────────────────────────────────────
    mlflow.log_param("model_type",   "RandomForestClassifier")
    mlflow.log_param("num_trees",    50)
    mlflow.log_param("max_depth",    6)
    mlflow.log_param("train_size",   train_df.count())
    mlflow.log_param("test_size",    test_df.count())
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("features",     str(feature_cols))

    # ── Train ─────────────────────────────────────────────
    print("⏳ Training model...")
    model = pipeline.fit(train_df)
    print("✅ Model trained!\n")

    # ── Predict ───────────────────────────────────────────
    predictions = model.transform(test_df)

    # ── Evaluate ──────────────────────────────────────────
    auc_eval = BinaryClassificationEvaluator(
        labelCol         = "is_fraud",
        rawPredictionCol = "rawPrediction",
        metricName       = "areaUnderROC"
    )
    auc = auc_eval.evaluate(predictions)

    acc_eval = MulticlassClassificationEvaluator(
        labelCol      = "is_fraud",
        predictionCol = "prediction",
        metricName    = "accuracy"
    )
    accuracy = acc_eval.evaluate(predictions)

    prec_eval = MulticlassClassificationEvaluator(
        labelCol      = "is_fraud",
        predictionCol = "prediction",
        metricName    = "weightedPrecision"
    )
    precision = prec_eval.evaluate(predictions)

    rec_eval = MulticlassClassificationEvaluator(
        labelCol      = "is_fraud",
        predictionCol = "prediction",
        metricName    = "weightedRecall"
    )
    recall = rec_eval.evaluate(predictions)

    # ── Log Metrics ───────────────────────────────────────
    mlflow.log_metric("auc_roc",   round(auc, 4))
    mlflow.log_metric("accuracy",  round(accuracy, 4))
    mlflow.log_metric("precision", round(precision, 4))
    mlflow.log_metric("recall",    round(recall, 4))

    # ── Tags ──────────────────────────────────────────────
    mlflow.set_tag("author",  "Zeeshan Ahmed")
    mlflow.set_tag("dataset", "fraud_db.silver_transactions")
    mlflow.set_tag("stage",   "development")

    # ── Save Predictions Table ────────────────────────────
    print("💾 Saving predictions table...")
    spark.sql("DROP TABLE IF EXISTS fraud_db.model_predictions")

    predictions.select(
        "amount",
        "hour_of_day",
        "login_attempts",
        "is_night_transaction",
        "is_high_amount",
        "is_suspicious_country",
        "is_multiple_attempts",
        "is_online_or_atm",
        "fraud_risk_score",
        "is_fraud",
        "prediction",
        vector_to_array(col("probability")).getItem(1).alias("fraud_probability")
    ).write \
     .format("delta") \
     .mode("overwrite") \
     .option("overwriteSchema", "true") \
     .saveAsTable("fraud_db.model_predictions")

    saved_rows = spark.table("fraud_db.model_predictions").count()
    mlflow.log_param("predictions_table", "fraud_db.model_predictions")
    mlflow.log_param("predictions_rows",  saved_rows)

    # ── Final Report ──────────────────────────────────────
    print("=" * 55)
    print("  📊 MODEL PERFORMANCE REPORT")
    print("=" * 55)
    print(f"  🎯 AUC-ROC    : {round(auc, 4)}   ← most important!")
    print(f"  ✅ Accuracy   : {round(accuracy*100, 2)}%")
    print(f"  🎯 Precision  : {round(precision, 4)}")
    print(f"  📡 Recall     : {round(recall, 4)}")
    print("=" * 55)
    print(f"\n  ✅ Params + Metrics  → logged to MLflow")
    print(f"  ✅ Predictions       → fraud_db.model_predictions")
    print(f"  ✅ Rows saved        → {saved_rows:,}")
    print(f"  🔑 Run ID            → {RUN_ID}")

In [0]:
# ============================================================
# 04_ML_FraudModel | CELL 5 — Feature Importance
# ============================================================

rf_model    = model.stages[-1]
importances = rf_model.featureImportances.toArray()

importance_data = sorted(
    zip(feature_cols, importances),
    key     = lambda x: x[1],
    reverse = True
)

print("🎯 FEATURE IMPORTANCE RANKING")
print("=" * 50)
print(f"  {'Feature':<30s}  Importance")
print("-" * 50)
for feat, imp in importance_data:
    bar = "█" * int(imp * 100)
    print(f"  {feat:<30s}  {round(imp,4):>6}  {bar}")
print("=" * 50)
print("\n💡 Higher = model relies on this feature MORE")

In [0]:

from pyspark.sql.functions import col, when, lit, round as spark_round
from pyspark.ml.functions import vector_to_array

# Read directly from saved Delta table — no Vector type issues!
pred_df = spark.table("fraud_db.model_predictions")

result_df = pred_df.withColumn(
    "verdict",
    when(
        (col("is_fraud") == 1) & (col("prediction") == 1), lit("✅ TRUE POSITIVE")
    ).when(
        (col("is_fraud") == 0) & (col("prediction") == 0), lit("✅ TRUE NEGATIVE")
    ).when(
        (col("is_fraud") == 0) & (col("prediction") == 1), lit("⚠️  FALSE POSITIVE")
    ).otherwise(lit("🚨 FALSE NEGATIVE"))
)

print("🔍 SAMPLE PREDICTIONS (20 rows):")
display(result_df.orderBy("is_fraud", ascending=False).limit(20))

print("\n📊 PREDICTION BREAKDOWN:")
result_df.groupBy("verdict").count() \
         .orderBy("count", ascending=False).show()

# Extra insight — avg fraud probability for actual fraud vs legit
print("📊 Avg Fraud Probability by Actual Label:")
result_df.groupBy("is_fraud") \
         .agg(
             spark_round(
                 __import__('pyspark.sql.functions', fromlist=['avg']).avg("fraud_probability"), 4
             ).alias("avg_fraud_probability")
         ).show()

In [0]:
# ============================================================
# 04_ML_FraudModel | CELL 7 — MLflow Run Summary
# ============================================================

from mlflow.tracking import MlflowClient

client = MlflowClient()
run    = client.get_run(RUN_ID)

print("=" * 55)
print("  ✅ MLflow Run Summary")
print("=" * 55)
print(f"  🔑 Run ID      : {RUN_ID}")
print(f"  📋 Status      : {run.info.status}")

print(f"\n  📊 Logged Metrics:")
for k, v in run.data.metrics.items():
    print(f"     {k:<20s} : {v}")

print(f"\n  ⚙️  Logged Params:")
for k, v in run.data.params.items():
    print(f"     {k:<20s} : {v}")

# Verify tables
tables = {
    "bronze_transactions" : "🪨 Bronze",
    "silver_transactions" : "🥈 Silver",
    "model_predictions"   : "🤖 Predictions",
}

print(f"\n  📦 Delta Tables:")
for table, label in tables.items():
    try:
        cnt = spark.table(f"fraud_db.{table}").count()
        print(f"     {label:<20s} → {cnt:,} rows  ✅")
    except:
        print(f"     {label:<20s} → ❌ NOT FOUND")

print("\n" + "=" * 55)
print("  🟢 Phase 4 COMPLETE!")
print("  🎯 Model Trained + Tracked in MLflow ✅")
print("  📦 Predictions saved in Delta Table  ✅")
print("  ⏭️  Next → Phase 5: Live Scoring Pipeline!")
print("=" * 55)